# Sentiment Analysis with Deep Learning using BERT

### Prerequisites

- Intermediate-level knowledge of Python 3 (NumPy and Pandas preferably, but not required)
- Exposure to PyTorch usage
- Basic understanding of Deep Learning and Language Models (BERT specifically)

### Project Outline

**Task 1**: Introduction (this section)

**Task 2**: Exploratory Data Analysis and Preprocessing

**Task 3**: Training/Validation Split

**Task 4**: Loading Tokenizer and Encoding our Data

**Task 5**: Setting up BERT Pretrained Model

**Task 6**: Creating Data Loaders

**Task 7**: Setting Up Optimizer and Scheduler

**Task 8**: Defining our Performance Metrics

**Task 9**: Creating our Training Loop

**Task 10**: Loading and Evaluating our Model

## Task 1: Introduction

### What is BERT

BERT is a large-scale transformer-based Language Model that can be finetuned for a variety of tasks.

For more information, the original paper can be found [here](https://arxiv.org/abs/1810.04805). 

[HuggingFace documentation](https://huggingface.co/transformers/model_doc/bert.html)

[Bert documentation](https://characters.fandom.com/wiki/Bert_(Sesame_Street) ;)

<img src="Images/BERT_diagrams.pdf" width="1000">

## Task 2: Exploratory Data Analysis and Preprocessing

We will use the SMILE Twitter dataset.

_Wang, Bo; Tsakalidis, Adam; Liakata, Maria; Zubiaga, Arkaitz; Procter, Rob; Jensen, Eric (2016): SMILE Twitter Emotion dataset. figshare. Dataset. https://doi.org/10.6084/m9.figshare.3187909.v2_

In [1]:
# Importing the libraries

import torch
import pandas as pd
from tqdm.notebook import tqdm

In [4]:
# Loading the dataset
df = pd.read_csv('Data/smile-annotations-final.csv',
                names = ['id', 'text', 'category' ])

In [5]:
# Setting id column as the index
df.set_index('id', inplace=True)

In [6]:
df.head()

,text,category
id,,
611857364396965889,@aandraous @britishmuseum @AndrewsAntonio Merc...,nocode
614484565059596288,Dorian Gray with Rainbow Scarf #LoveWins (from...,happy
614746522043973632,@SelectShowcase @Tate_StIves ... Replace with ...,happy
614877582664835073,@Sofabsports thank you for following me back. ...,happy
611932373039644672,@britishmuseum @TudorHistory What a beautiful ...,happy


In [7]:
# Checking the different emotions present in the category column
df.category.value_counts()

#nocode - no clear emotion, so we ignore this

nocode               1572
happy                1137
not-relevant          214
angry                  57
surprise               35
sad                    32
happy|surprise         11
happy|sad               9
disgust|angry           7
disgust                 6
sad|disgust             2
sad|angry               2
sad|disgust|angry       1
Name: category, dtype: int64

In [26]:
# Removing the tweets with multiple emotions and nocode values
# Since "|" is a special symbol, we have to use "\" to identify it in the string

df = df[(df['category'] != 'nocode') & ~(df['category'].str.contains('\|'))]

In [27]:
# There is a class imbalance as the number of happy tweets are much higher than others
df.category.value_counts()

happy           1137
not-relevant     214
angry             57
surprise          35
sad               32
disgust            6
Name: category, dtype: int64

In [29]:
category_dict = {}
for i, emotion in enumerate(df['category'].unique()):
    category_dict[emotion] = i
category_dict

{'happy': 0,
 'not-relevant': 1,
 'angry': 2,
 'disgust': 3,
 'sad': 4,
 'surprise': 5}

In [31]:
df['category_labels'] = df['category'].apply(lambda x: category_dict[x])

In [32]:
df.head()

,text,category,category_labels
id,,,
614484565059596288,Dorian Gray with Rainbow Scarf #LoveWins (from...,happy,0
614746522043973632,@SelectShowcase @Tate_StIves ... Replace with ...,happy,0
614877582664835073,@Sofabsports thank you for following me back. ...,happy,0
611932373039644672,@britishmuseum @TudorHistory What a beautiful ...,happy,0
611570404268883969,@NationalGallery @ThePoldarkian I have always ...,happy,0


## Task 3: Training/Validation Split

In [33]:
from sklearn.model_selection import train_test_split

In [36]:
X_train, X_val, y_train, y_val = train_test_split(
    df.index.values,
    df.category_labels.values,
    test_size = 0.15,
    random_state = 17,
    stratify = df.category_labels.values)

In [37]:
df['data_type'] = ['not_set'] * df.shape[0]

In [39]:
df.loc[X_train, 'data_type'] = 'train'

df.loc[X_val, 'data_type'] = 'val'

In [42]:
df.groupby(['category', 'category_labels', 'data_type']).count()

text
category     category_labels data_type      
angry        2               train        48
                             val           9
disgust      3               train         5
                             val           1
happy        0               train       966
                             val         171
not-relevant 1               train       182
                             val          32
sad          4               train        27
                             val           5
surprise     5               train        30
                             val           5

## Task 4: Loading Tokenizer and Encoding our Data

In [43]:
from transformers import BertTokenizer
from torch.utils.data import TensorDataset

In [44]:
# from_pretrained because we are using pretrained model - uncased means lower cased
# do_lower is set to True since we are using uncased version
tokenizer = BertTokenizer.from_pretrained(
    'bert-base-uncased',
    do_lower = True
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

#### Understanding the parameters

1. <b>add_special_tokens=True</b>

    Tells the tokenizer to include BERT-style special markers.
    For BERT:
    - [CLS] at the start
    - [SEP] at the end
    - [PAD] if padded
    
        Example Text: "Hello world"
        
        Without special tokens →
        ['hello', 'world']
        
        With special tokens →
        ['[CLS]', 'hello', 'world', '[SEP]']
    
2. <b>return_attention_mask = True    </b>

    Creates an attention mask:

    - 1 for real tokens

    - 0 for padded tokens

        The model uses this to ignore padding during training.

        Example: Suppose your sentence is padded to length 6:

        Tokens: ['[CLS]', 'hello', 'world', '[SEP]', '[PAD]', '[PAD]']
        Attention mask: [1, 1, 1, 1, 0, 0]
        
        
3. <b>pad_to_max_length = True </b>

    Pads all sequences to the same fixed length (max_length).

    This is required to batch sequences together.

        Example: max_length = 6

        Sentence A token ids: [101, 7592, 2088, 102] → becomes [101, 7592, 2088, 102, 0, 0]
        Sentence B token ids (longer) might get truncated instead.
        
4. <b>max_length = 256</b>


    Sets the maximum number of tokens:

    - Shorter sequences → padded

    - Longer sequences → truncated

        Example: Text tokenized length = 300 → truncated to 256 tokens
        Padding applied only if < 256
        
5. <b>return_tensors='pt'</b>

    Returns PyTorch tensors instead of Python lists.

In [47]:
#Encoding training data
encoded_data_train = tokenizer.batch_encode_plus(
    df[df['data_type'] == 'train']['text'].values,
    add_special_tokens = True,
    return_attention_mask = True,
    pad_to_max_length = True,
    max_length = 256,
    return_tensors = 'pt'
)


#Encoding validation data
encoded_data_val = tokenizer.batch_encode_plus(
    df[df['data_type'] == 'val']['text'].values,
    add_special_tokens = True,
    return_attention_mask = True,
    pad_to_max_length = True,
    max_length = 256,
    return_tensors = 'pt'
)

In [48]:
# Encoded data is in a form of a dictonairy. We need to extract input_ids, attention masks and labels to feed into our model
encoded_data_train

{'input_ids': tensor([[  101, 16092,  3897,  ...,     0,     0,     0],
        [  101,  1030, 27034,  ...,     0,     0,     0],
        [  101,  1030, 10682,  ...,     0,     0,     0],
        ...,
        [  101, 11047,  1030,  ...,     0,     0,     0],
        [  101,  1030,  3680,  ...,     0,     0,     0],
        [  101,  1030,  2120,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

In [49]:
input_ids_train = encoded_data_train['input_ids']
attention_masks_train = encoded_data_train['attention_mask']
labels_train = torch.tensor(df[df['data_type'] == 'train'].category_labels.values)


input_ids_val = encoded_data_val['input_ids']
attention_masks_val = encoded_data_val['attention_mask']
labels_val = torch.tensor(df[df['data_type'] == 'val'].category_labels.values)

In [50]:
dataset_train = TensorDataset(input_ids_train,attention_masks_train, labels_train )

dataset_val = TensorDataset(input_ids_val,attention_masks_val, labels_val )

In [52]:
len(dataset_train), len(dataset_val)

(1258, 223)

## Task 5: Setting up BERT Pretrained Model

In [53]:
from transformers import BertForSequenceClassification

In [55]:
# Loads:
# pretrained BERT weights
# replaces last layer with classifier head of size num_labels
# does NOT return attention maps
# does NOT return hidden state


model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels = len(category_dict),
    output_attentions=False,
    output_hidden_states = False
                                     )

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Task 6: Creating Data Loaders

- DataLoader returns one batch at a time:

    for batch in dataloader:
    
        ...



- Without DataLoader you’d need to manually split:

    for i in range(0, len(dataset), batch_size):
    
        batch = dataset[i:i+batch_size]

In [56]:
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

In [57]:
batch_size = 4 #32 but due to limited compute we use 4

# takes samples from dataset_train
# shuffles them each epoch
# groups them into batches of batch_size
# yields tensors ready for training

# RandomSampler ensures:
# Each epoch → new random order
# No samples are skipped or repeated

# SequentialSampler - Iterate in order (useful for eval)
# WeightedRandomSampler - Sampling with weights (fix class imbalance)


dataloader_train = DataLoader(
    dataset_train,
    sampler = RandomSampler(dataset_train),
    batch_size = batch_size
)

dataloader_val = DataLoader(
    dataset_val,
    sampler = RandomSampler(dataset_val),
    batch_size = 32  #since gradients are fixed and no backpropagation is required here
)


## Task 7: Setting Up Optimizer and Scheduler

⚙️ Adam vs AdamW

AdamW is an improved version of Adam.

🔹 Adam: Adaptive Moment Estimation

    Adam adjusts learning rate per parameter based on:

        - mean of gradients (momentum)

        - variance of gradients

        - Makes training:

            faster, smoother and less sensitive to LR choice

But Adam has an issue with weight decay.

🛑 The Problem AdamW Solves

    In regular Adam, L2 regularization (weight decay) gets mixed into the gradient update incorrectly.

    This causes:

        - Over-regularization

        - Worse generalization

        - Too much shrinking of weights

In [58]:
from transformers import AdamW, get_linear_schedule_with_warmup

In [59]:
# eps = 1e-8
#  A tiny number added to the denominator to avoid division by zero.
#  This affects the internal Adam calculation:
#       update = lr * m / (sqrt(v) + eps)
#       If v (variance estimate) becomes very small, eps prevents blowing up the update


optimizer = AdamW(
    model.parameters(),
    lr = 1e-5,
    eps = 1e-8,
)

In [60]:
# A scheduler changes the learning rate during training instead of keeping it constant.


# What does linear schedule do?
# Learning rate follows a linearly decreasing curve:
#       start_lr ——→ gradually drops ——→ 0 at final step

        # Since warmup=0:

        # Step 0: lr = 1e-5
        # Step halfway: lr = ~5e-6
        # End: lr ≈ 0


# Why decrease LR?
#     1. early: explore and learn aggressively
#     2. late: fine-tune carefully without overshooting minima



epochs = 10
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps = 0,
    num_training_steps = len(dataloader_train)*epochs



)

## Task 8: Defining our Performance Metrics

Accuracy metric approach originally used in accuracy function in [this tutorial](https://mccormickml.com/2019/07/22/BERT-fine-tuning/#41-bertforsequenceclassification).

In [61]:
import numpy as np

In [62]:
from sklearn.metrics import f1_score

In [67]:
def f1_score_func(preds, labels):
    preds_flat = np.argmax(preds, axis=1).flatten() # convert lists of lists to a single list
    labels_flat = labels.flatten()
    return f1_score(preds_flat,  labels_flat, average='weighted')

In [68]:
# argmax picks the class with highest predicted score


def accuracy_per_class(preds, labels):
    category_dict_inverse = {v:k for k,v in category_dict.items()}
    
    preds_flat = np.argmax(preds, axis=1).flatten() # convert lists of lists to a single list
    labels_flat = labels.flatten()
    
    for label in np.unique(labels_flat):
        y_preds = preds_flat[labels_flat==label] #only select value with this label
        y_true = labels_flat[labels_flat == label]
        print(f'Class: {category_dict_inverse[label]}')
        print(f'Accuracy: {len(y_pred[y_preds == label])/len(y_true)}\n')
    
    
    
    

## Task 9: Creating our Training Loop

Approach adapted from an older version of HuggingFace's `run_glue.py` script. Accessible [here](https://github.com/huggingface/transformers/blob/5bfcd0485ece086ebcbed2d008813037968a9e58/examples/run_glue.py#L128).

In [69]:
import random

seed_val = 17
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val) #for gpu

In [70]:
# Check if GPU is available, otherwise set device to cpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model.to(device)

print(device)

cpu


In [71]:
def evaluate(dataloader_val):

    model.eval()
    
    loss_val_total = 0
    predictions, true_vals = [], []
    
    for batch in dataloader_val:
        
        batch = tuple(b.to(device) for b in batch)
        
        inputs = {'input_ids':      batch[0],
                  'attention_mask': batch[1],
                  'labels':         batch[2],
                 }

        with torch.no_grad():        
            outputs = model(**inputs)
            
        loss = outputs[0]
        logits = outputs[1]
        loss_val_total += loss.item()

        logits = logits.detach().cpu().numpy()
        label_ids = inputs['labels'].cpu().numpy()
        predictions.append(logits)
        true_vals.append(label_ids)
    
    loss_val_avg = loss_val_total/len(dataloader_val) 
    
    predictions = np.concatenate(predictions, axis=0)
    true_vals = np.concatenate(true_vals, axis=0)
            
    return loss_val_avg, predictions, true_vals


In [78]:
for epoch in tqdm(range(1, epochs+1)):
    
    
    model.train()   #set model to training mode
#     Turns on:
#         1. Dropout layers
#         2. LayerNorm behaviors that differ from eval
    
    
    loss_train_total = 0
    
    progress_bar = tqdm(dataloader_train, desc = 'Epoch {:1d}'.format(epoch),
                       leave=False, disable=False)
    
    
    for batch in progress_bar:
        
        model.zero_grad()   #setting inital gradients to 0
#       Clears gradients left over from the previous batch.
#       PyTorch accumulates gradients by default — if you forget this, training explodes.
        
        batch = tuple(b.to(device) for b in batch)
        #Without this, your model (on GPU) and data (on CPU) would mismatch.
        
        inputs = {
            'input_ids'       : batch[0],
            'attention_mask' : batch[1],
            'labels'          : batch[2]
        
        }
        
        #forward pass
        outputs = model(**inputs)
        
#         With labels provided, outputs includes:
#               1. loss
#               2. logits
        
        loss = outputs[0]
        loss_train_total += loss.item()  #.item() converts tensor → float so we can sum it.
        loss.backward() #Compute gradients
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) #Prevents exploding gradients. If gradients > 1.0, they are scaled down.
        
        
        optimizer.step() # update model weights
        scheduler.step() # update learning rate
        
        progress_bar.set_postfix({'training_loss': '{:.3f}'.format(loss.item()/len(batch))})
        
        
        
    torch.save(model.state_dict(), f'Models/Bert_ft_epoch{epoch}.model') #saving model for each epoch
    
    tqdm.write('\nEpoch {epoch}')
    
    loss_train_avg = loss_train_total/len(dataloader)
    tqdm.write(f'Training Loss: {loss_train_avg}')
    
    val_loss, predictions, true_val = evaluate(dataloader_val)
    val_f1 = f1_score_func(predictions, true_vals)
    tqdm.write(f'Validation_loss: {val_loss}')
    tqdm.write(f'F1 score (weighted): {val_f1}')

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1:   0%|          | 0/315 [00:00<?, ?it/s]

KeyboardInterrupt: 

SyntaxError: invalid syntax (884539294.py, line 1)

## Task 10: Loading and Evaluating our Model

In [14]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased",
                                                      num_labels=len(label_dict),
                                                      output_attentions=False,
                                                      output_hidden_states=False)

NameError: name 'label_dict' is not defined

In [ ]:
model.to(device)
pass